In [3]:
import pandas as pd

In [120]:
telemetry_r1_parquet_path = "barber-motorsports-park/barber/R1_barber_telemetry_data.parquet"
telemetry_r2_parquet_path = "barber-motorsports-park/barber/R2_barber_telemetry_data.parquet"

tele_R1 = pd.read_parquet(telemetry_r1_parquet_path)
tele_R2 = pd.read_parquet(telemetry_r2_parquet_path)

In [121]:
df = pd.concat([tele_R1,tele_R2])

In [122]:
df = df[~df['vehicle_number'].isin([16,0,78])]


In [123]:
weather_race_1_path = "barber-motorsports-park/barber/26_Weather_Race 1_Anonymized.CSV"
weather_race_2_path = "barber-motorsports-park/barber/26_Weather_Race 2_Anonymized.CSV"

In [124]:
weather_race_1 = pd.read_csv(weather_race_1_path, sep = ';')
weather_race_2 = pd.read_csv(weather_race_2_path, sep = ';')

In [125]:
weather = pd.concat([weather_race_1,weather_race_2])



In [126]:
# this is copied from wind_analysis.ipynb

import pandas as pd, numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import QuantileRegressor

# ------------ Geometry helpers ------------
def to_local_xy(lat_deg, lon_deg):
    lat0 = np.deg2rad(np.nanmedian(lat_deg)); lon0 = np.deg2rad(np.nanmedian(lon_deg))
    R = 6_371_000.0
    lat = np.deg2rad(lat_deg); lon = np.deg2rad(lon_deg)
    x = (lon - lon0) * np.cos(lat0) * R  # meters
    y = (lat - lat0) * R                 # meters
    return x, y

# ------------ Telemetry (outputs kph; keeps s_m for reference only) ------------
def prep_telemetry_kph(df_long,
                       time_col='meta_time', name_col='telemetry_name', value_col='telemetry_value',
                       lat_name='VBOX_Lat_Min', lon_name='VBOX_Long_Minutes', speed_name='speed',
                       resample='1S', max_seg_m=120):
    tele = (df_long
            .pivot_table(index=time_col, columns=name_col, values=value_col, aggfunc='median')
            .sort_index())
    tele.index = pd.to_datetime(tele.index, utc=True, errors='coerce')
    tele = tele[~tele.index.isna()].copy()

    need = [lat_name, lon_name, speed_name]
    miss = [c for c in need if c not in tele.columns]
    if miss: raise ValueError(f"Missing telemetry channels: {miss}")
    for c in need: tele[c] = pd.to_numeric(tele[c], errors='coerce')

    if resample: tele = tele.resample(resample).median()
    tele = tele.dropna(subset=[lat_name, lon_name]).copy()

    tele['lat_deg'] = tele[lat_name].astype(float)
    tele['lon_deg'] = tele[lon_name].astype(float)
    tele['speed_kph'] = tele[speed_name].astype(float)

    # heading & unit tangent
    lat = np.deg2rad(tele['lat_deg']); lon = np.deg2rad(tele['lon_deg'])
    dlat, dlon = lat.diff(), lon.diff()
    y = np.sin(dlon) * np.cos(lat)
    x = np.cos(lat.shift())*np.sin(lat) - np.sin(lat.shift())*np.cos(lat)*np.cos(dlon)
    heading = np.arctan2(y, x).fillna(method='bfill')
    tele['heading_rad'] = heading
    tele['heading_deg'] = np.rad2deg(heading)
    tele['ex'] = np.sin(heading); tele['ey'] = np.cos(heading)

    # along-track distance (meters) with spike guard
    R = 6_371_000.0
    a = np.sin(dlat/2)**2 + np.cos(lat)*np.cos(lat.shift())*np.sin(dlon/2)**2
    seg = 2*R*np.arcsin(np.sqrt(a)); seg.iloc[0] = 0.0
    seg = seg.where(np.isfinite(seg), 0.0).clip(lower=0.0).mask(seg > max_seg_m, 0.0)
    tele['s_m'] = seg.cumsum()

    return tele[['lat_deg','lon_deg','speed_kph','heading_rad','heading_deg','ex','ey','s_m']]

# ------------ Weather → kph + components (east/north in kph) ------------
def prep_wind_kph(df_wind,
                  time_col='TIME_UTC_STR', speed_col='WIND_SPEED', dir_col='WIND_DIRECTION',
                  orig_unit='mps'):  # if your raw is 'kph' or 'knots', set here
    wx = df_wind[[time_col, speed_col, dir_col]].copy()
    wx['time'] = pd.to_datetime(wx[time_col], utc=True, errors='coerce')
    wx = wx.dropna(subset=['time']).sort_values('time')

    spd = pd.to_numeric(wx[speed_col], errors='coerce')
    if orig_unit == 'mps':
        spd_kph = spd * 3.6
    elif orig_unit == 'knots':
        spd_kph = spd * 1.852
    else:  # already kph
        spd_kph = spd
    wx['ws_kph'] = spd_kph

    theta = np.deg2rad(pd.to_numeric(wx[dir_col], errors='coerce'))  # meteorological FROM, CW from North
    wx['u_e_kph'] = -wx['ws_kph'] * np.sin(theta)  # east
    wx['v_n_kph'] = -wx['ws_kph'] * np.cos(theta)  # north
    return wx[['time','ws_kph','u_e_kph','v_n_kph']]

# ------------ Join + wind projections (all kph) ------------
def join_wind_to_telemetry_kph(tele_kph, wx_kph, tolerance='90s'):
    t = tele_kph.copy()
    t['meta_time'] = t.index
    m = pd.merge_asof(
        t.sort_values('meta_time'),
        wx_kph.sort_values('time'),
        left_on='meta_time', right_on='time',
        direction='nearest', tolerance=pd.Timedelta(tolerance)
    )
    # project wind onto car’s tangent; outputs in kph
    m['headwind_kph']  = m['u_e_kph']*m['ex'] + m['v_n_kph']*m['ey']
    m['crosswind_kph'] = m['u_e_kph']*m['ey'] - m['v_n_kph']*m['ex']
    keep = ['meta_time','lat_deg','lon_deg','s_m','speed_kph','ex','ey',
            'ws_kph','u_e_kph','v_n_kph','headwind_kph','crosswind_kph']
    keep = [c for c in keep if c in m.columns]
    return m[keep].set_index('meta_time')

# ------------ Spatial index + local fast-speed mapping (kph) ------------
def build_spatial_index(tw_kph):
    d = tw_kph[['lat_deg','lon_deg','speed_kph','headwind_kph','crosswind_kph']].dropna().copy()
    d['x_m'], d['y_m'] = to_local_xy(d['lat_deg'].to_numpy(), d['lon_deg'].to_numpy())
    d['abs_cross_kph'] = d['crosswind_kph'].abs()
    nn = NearestNeighbors(n_neighbors=min(400, len(d)), algorithm='kd_tree').fit(d[['x_m','y_m']].to_numpy())
    return d, nn

def local_ideal_speed_kph(d, nn, x0, y0, hw_kph, cw_kph, k=200, radius_m=80):
    dist, idx = nn.kneighbors(np.array([[x0, y0]]), n_neighbors=min(k, len(d)), return_distance=True)
    dist, idx = dist[0], idx[0]
    mask = dist <= radius_m
    if mask.sum() < 40:
        mask = np.arange(len(idx)) < min(max(80, mask.sum()*2), len(idx))
    g = d.iloc[idx[mask]]
    if len(g) < 25:
        return np.nan
    X = g[['headwind_kph','abs_cross_kph']].to_numpy()
    y = g['speed_kph'].to_numpy()
    qr = QuantileRegressor(quantile=0.9, alpha=0.0).fit(X, y)
    return float(qr.predict([[hw_kph, abs(cw_kph)]]))  # kph

def add_ideal_speed_spatial_kph(tw_kph, d, nn, k=200, radius_m=80):
    x, y = to_local_xy(tw_kph['lat_deg'].to_numpy(), tw_kph['lon_deg'].to_numpy())
    preds = [local_ideal_speed_kph(d, nn, xi, yi, hw, cw, k=k, radius_m=radius_m)
             for xi, yi, hw, cw in zip(x, y, tw_kph['headwind_kph'], tw_kph['crosswind_kph'])]
    out = tw_kph.copy()
    out['ideal_speed_kph'] = preds
    return out

# ------------ Usage ------------
# tele_long: ['telemetry_value','telemetry_name','meta_time']
# df_wind:   ['TIME_UTC_STR','WIND_SPEED','WIND_DIRECTION']

tele_prep = prep_telemetry_kph(df, resample='1S')                    # all speeds in kph
wx_prep   = prep_wind_kph(weather, orig_unit='mps')                         # convert raw → kph
# tw        = join_wind_to_telemetry_kph(tele_prep, wx_prep, tolerance='90s') # adds head/crosswind_kph

# # build spatial model and compute local fast (p90) achievable speed in kph
# d, nn     = build_spatial_index(tw.query("speed_kph > 18"))  # >18 kph ≈ >5 m/s to drop pits/crawling
# tw_map    = add_ideal_speed_spatial_kph(tw, d, nn, k=200, radius_m=80)

# tw_map now has: speed_kph, ws/u_e/v_n/headwind/crosswind all in kph, plus ideal_speed_kph


C:\Users\user\AppData\Local\Temp\ipykernel_16992\4154775357.py:32: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  if resample: tele = tele.resample(resample).median()
C:\Users\user\AppData\Local\Temp\ipykernel_16992\4154775357.py:44: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  heading = np.arctan2(y, x).fillna(method='bfill')
C:\Users\user\AppData\Local\Temp\ipykernel_16992\4154775357.py:63: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  wx['time'] = pd.to_datetime(wx[time_col], utc=True, errors='coerce')


In [127]:
import pandas as pd
import numpy as np

# assumes you already have:
# tele_prep = prep_telemetry_kph(tele_long, resample=None or '1S')  # index = meta_time (UTC)
# wx_prep   = prep_wind_kph(df_wind, orig_unit='mps')               # columns: time, ws_kph, u_e_kph, v_n_kph

def join_wind_same_minute(tele_kph: pd.DataFrame, wx_kph: pd.DataFrame) -> pd.DataFrame:
    t = tele_kph.copy()
    w = wx_kph.copy()

    # 1) Add “minute” keys (floor to minute)
    t['minute'] = t.index.floor('T')
    w['minute'] = pd.to_datetime(w['time'], utc=True, errors='coerce').dt.floor('T')

    # 2) Reduce weather to exactly one row per minute (mean within the minute)
    w_min = (w.groupby('minute', as_index=False)
               .agg(ws_kph=('ws_kph','mean'),
                    u_e_kph=('u_e_kph','mean'),
                    v_n_kph=('v_n_kph','mean')))

    # 3) Strict minute-level join (no tolerance)
    m = (t.reset_index()
           .merge(w_min, on='minute', how='left')
           .rename(columns={'index':'meta_time'}))

    # 4) Project wind onto car’s tangent (all kph)
    m['headwind_kph']  = m['u_e_kph']*m['ex'] + m['v_n_kph']*m['ey']
    m['crosswind_kph'] = m['u_e_kph']*m['ey'] - m['v_n_kph']*m['ex']

    keep = ['meta_time','lat_deg','lon_deg','s_m','speed_kph','ex','ey',
            'ws_kph','u_e_kph','v_n_kph','headwind_kph','crosswind_kph']
    keep = [c for c in keep if c in m.columns]
    return m[keep].set_index('meta_time').sort_index()

# ---- usage ----
tw = join_wind_same_minute(tele_prep, wx_prep)


C:\Users\user\AppData\Local\Temp\ipykernel_16992\4285224910.py:13: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  t['minute'] = t.index.floor('T')
C:\Users\user\AppData\Local\Temp\ipykernel_16992\4285224910.py:14: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  w['minute'] = pd.to_datetime(w['time'], utc=True, errors='coerce').dt.floor('T')


In [128]:
tw_clean = tw[~tw['ws_kph'].isna()]

In [ ]:


twA = tw_clean.dropna(subset=['speed_kph','headwind_kph','crosswind_kph','lat_deg','lon_deg']).copy()
twA = twA.query('speed_kph > 60')  # keep race pace
twA['x_m'], twA['y_m'] = to_local_xy(twA['lat_deg'].to_numpy(), twA['lon_deg'].to_numpy())
twA['abs_cross_kph'] = twA['crosswind_kph'].abs()


# (optional) simple interaction to capture different wind effect at different places
twA['x_head'] = twA['x_m'] * twA['headwind_kph']
twA['y_head'] = twA['y_m'] * twA['headwind_kph']

# ---- Global “fast-achievable” model via 90th percentile ----
feat = ['headwind_kph','abs_cross_kph','x_m','y_m','x_head','y_head']

X, y = twA[feat], twA['speed_kph']


In [130]:
import pickle
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score


with open('best_xgb.pkl', 'rb') as f:
    best_xgb = pickle.load(f)

In [131]:
pred_xgb = best_xgb.predict(X)

print("XGBoost MAE:", round(mean_absolute_error(twA['speed_kph'], pred_xgb), 2),
      "R²:", round(r2_score(twA['speed_kph'], pred_xgb), 3))

XGBoost MAE: 16.08 R²: -0.448


In [115]:
twA

,lat_deg,lon_deg,s_m,speed_kph,ex,ey,ws_kph,u_e_kph,v_n_kph,headwind_kph,crosswind_kph,x_m,y_m,abs_cross_kph,x_head,y_head
meta_time,,,,,,,,,,,,,,,,
2025-09-07 15:05:02+00:00,33.531019,-86.618999,0.000000,111.3450,-0.917804,-0.397034,18.144,3.772350,-17.74751,3.584090,-17.786485,-14.143261,-141.038182,17.786485,-50.690720,-505.493541
2025-09-07 15:05:04+00:00,33.530792,-86.619629,63.566915,109.2100,-0.917804,-0.397034,18.144,3.772350,-17.74751,3.584090,-17.786485,-72.484211,-166.276593,17.786485,-259.789937,-595.950279
2025-09-07 15:05:06+00:00,33.530381,-86.620760,177.935380,111.2600,-0.916680,-0.399623,18.144,3.772350,-17.74751,3.634280,-17.776297,-177.321131,-211.981448,17.776297,-644.434557,-770.399838
2025-09-07 15:05:08+00:00,33.530037,-86.621346,244.353654,112.0700,-0.817187,-0.576372,18.144,3.772350,-17.74751,7.146453,-16.677318,-231.595893,-250.263240,16.677318,-1655.089110,-1788.494419
2025-09-07 15:05:10+00:00,33.529688,-86.621675,293.768781,99.9800,-0.618955,-0.785427,18.144,3.772350,-17.74751,11.604456,-13.947808,-262.180695,-289.075251,13.947808,-3042.464302,-3354.560991
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-09-07 15:49:53+00:00,33.531330,-86.622074,70076.267005,141.0200,-0.991544,0.129767,15.552,1.355446,-15.49282,-3.354445,-15.185928,-299.129963,-106.467921,15.185928,1003.414930,357.140756
2025-09-07 15:49:54+00:00,33.531532,-86.619762,70076.267005,90.0350,0.994540,0.104357,15.552,1.355446,-15.49282,-0.268745,15.549678,-84.859564,-83.986647,15.549678,22.805607,22.571015
2025-09-07 15:49:55+00:00,33.530888,-86.622429,70076.267005,124.6600,-0.960422,-0.278549,15.552,1.355446,-15.49282,3.013702,-15.257205,-332.013044,-155.672219,15.257205,-1000.588431,-469.149703


In [136]:
pd.Series(pred_xgb)

0       113.228683
1       129.111633
2       144.085663
3       148.067352
4       153.150024
           ...    
2806    113.217514
2807     84.564674
2808    127.972397
2809    116.979759
2810    156.548630
Length: 2811, dtype: float32

In [134]:
df

,expire_at,lap,meta_event,meta_session,meta_source,meta_time,original_vehicle_id,outing,telemetry_name,telemetry_value,timestamp,vehicle_id,vehicle_number
1177440,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:49.968Z,GR86-006-7,0,accx_can,0.333000,2025-09-04T23:33:38.504Z,GR86-006-7,7
1177441,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:49.968Z,GR86-006-7,0,accy_can,0.120000,2025-09-04T23:33:38.504Z,GR86-006-7,7
1177442,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:49.968Z,GR86-006-7,0,aps,100.000000,2025-09-04T23:33:38.504Z,GR86-006-7,7
1177443,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:49.968Z,GR86-006-7,0,pbrake_r,0.000000,2025-09-04T23:33:38.504Z,GR86-006-7,7
1177444,NaN,1,I_R06_2025-09-07,R1,kafka:gr-raw,2025-09-06T18:40:49.968Z,GR86-006-7,0,pbrake_f,0.000000,2025-09-04T23:33:38.504Z,GR86-006-7,7
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11749599,NaN,28,I_R06_2025-09-07,R2,kafka:gr-raw,2025-09-07T15:48:40.654Z,GR86-065-5,0,gear,4.000000,2025-09-05T04:21:10.686Z,GR86-065-5,5
11749600,NaN,28,I_R06_2025-09-07,R2,kafka:gr-raw,2025-09-07T15:48:40.654Z,GR86-065-5,0,VBOX_Long_Minutes,-86.619598,2025-09-05T04:21:10.686Z,GR86-065-5,5
11749601,NaN,28,I_R06_2025-09-07,R2,kafka:gr-raw,2025-09-07T15:48:40.654Z,GR86-065-5,0,VBOX_Lat_Min,33.532635,2025-09-05T04:21:10.686Z,GR86-065-5,5
11749602,NaN,28,I_R06_2025-09-07,R2,kafka:gr-raw,2025-09-07T15:48:40.654Z,GR86-065-5,0,Steering_Angle,0.100000,2025-09-05T04:21:10.686Z,GR86-065-5,5
